In [ ]:
# ─── Cell 1: Install only Gradio & verify existing packages ───────────────────────
# Colab already includes numpy, pandas, matplotlib, pillow, and tensorflow,
# so we only need to install Gradio.
!pip install gradio==3.36.0 --quiet

# Verify versions of all key libraries
import numpy as np, pandas as pd, matplotlib, PIL, tensorflow as tf, gradio
print(f"▶ numpy      {np.__version__}")
print(f"▶ pandas     {pd.__version__}")
print(f"▶ matplotlib {matplotlib.__version__}")
print(f"▶ Pillow     {PIL.__version__}")
print(f"▶ tensorflow {tf.__version__}")
print(f"▶ gradio     {gradio.__version__}")


In [ ]:
# ─── Cell 2 (UPDATED): Imports, Seeding & Label Names ───────────────────────────
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import gradio as gr
import random
from tensorflow.keras.models import load_model

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# User-friendly class names for FashionMNIST labels 0–9
LABEL_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

print(f"▶ Random seed set to {SEED}")
print("▶ Label mapping:", {i: name for i, name in enumerate(LABEL_NAMES)})


In [4]:
# ─── Cell 3: Load Data & Quick Inspect ────────────────────────────────────────────
# Load the CSV files from Colab's local filesystem
train_df = pd.read_csv('fashion-mnist_train.csv')
test_df  = pd.read_csv('fashion-mnist_test.csv')

In [ ]:
# 1) Print dataset shapes
print(f"▶ train_df.shape = {train_df.shape}")
print(f"▶ test_df.shape  = {test_df.shape}")


In [ ]:
# 2) Peek at column names and first row
print("▶ train_df columns:", train_df.columns.tolist())
print("▶ train_df first row:\n", train_df.head(1))

In [ ]:
# 3) Check for any missing values
missing_train = train_df.isnull().sum().sum()
print(f"▶ Missing values in train_df: {missing_train}")

# 4) Label distribution in the training set
label_counts = train_df['label'].value_counts().sort_index()
print("▶ Label distribution:\n", label_counts)

# 5) Pixel intensity range across all pixels
pixel_min = train_df.iloc[:,1:].values.min()
pixel_max = train_df.iloc[:,1:].values.max()
print(f"▶ Pixel value range: [{pixel_min}, {pixel_max}]")


In [ ]:
# 6) Visualize one example image
example = train_df.iloc[0]
img_arr  = example.values[1:].astype(np.uint8).reshape(28,28)
plt.figure(figsize=(3,3))
plt.imshow(img_arr, cmap='gray')
plt.title(f"Label = {example[0]}")
plt.axis('off')
plt.show()

In [ ]:
# ─── Cell 4: Preprocess & Split ───────────────────────────────────────────────────
# Normalize pixel values to [0,1] and separate features/labels
X        = train_df.iloc[:,1:].values.astype('float32') / 255.0
y        = train_df.iloc[:,0].values.astype('int32')
X_test   = test_df.iloc[:,1:].values.astype('float32') / 255.0
y_test   = test_df.iloc[:,0].values.astype('int32')

# Shuffle the training data
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]
print(f"▶ First 5 labels after shuffling: {y[:5]}")

# Split into 90% train / 10% validation
split_idx    = int(0.9 * len(X))
X_train, y_train = X[:split_idx], y[:split_idx]
X_val,   y_val   = X[split_idx:], y[split_idx:]
print(f"▶ Split sizes → train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")
# ─── End of Cell 4 ───────────────────────────────────────────────────────────────


In [ ]:
# ─── Cell 5: Build & Compile TensorFlow Model ────────────────────────────────────
# Define a simple feed-forward neural network with one hidden layer
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(784,)),      # 28x28 flattened
    tf.keras.layers.Dense(128, activation='relu'),       # Hidden layer
    tf.keras.layers.Dense(10, activation='softmax'),     # Output layer
])

# Compile with Adam optimizer and cross-entropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display the model summary
model.summary()
# ─── End of Cell 5 ───────────────────────────────────────────────────────────────


In [ ]:
# ─── Cell 6: Train & Plot Curves ──────────────────────────────────────────────────
# Train for 20 epochs with validation
EPOCHS = 20
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    verbose=2
)


In [ ]:
# Plot training & validation loss
plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss over Epochs')
plt.grid(True)
plt.show()

# Plot training & validation accuracy
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy over Epochs')
plt.grid(True)
plt.show()


In [ ]:
# ─── Cell 7 (UPDATED): Evaluate on Test Set & Visualize Samples ─────────────────
# Evaluate final performance on the test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.4f}")


In [ ]:
# ─── Cell 8: Save Trained Model with Proper Extension ──────────────────────
# TensorFlow requires a file extension; use the native Keras format (.keras)
MODEL_PATH = 'fashion_mnist_saved_model.keras'

# Save the model
model.save(MODEL_PATH)

print(f"▶ Model successfully saved to '{MODEL_PATH}'")
# Use: loaded_model = tf.keras.models.load_model(MODEL_PATH) to reload later
# ─── End of Updated Cell ────────────────────────────────────────────────────────


In [ ]:
# ─── Cell 9: Load Saved Model for Inference ────────────────────────────────────

# Define path to the saved model directory
MODEL_DIR = 'fashion_mnist_saved_model.keras'

# Load the model
trained_model = load_model(MODEL_DIR)
print(f"▶ Model loaded from '{MODEL_DIR}'")

# (Optional) Verify by showing its architecture
trained_model.summary()
# ─── End of New Cell ─────────────────────────────────────────────────────────────


In [ ]:
# Visualize 5 random test images with Predicted vs. Actual labels (using names)
idxs = np.random.choice(len(X_test), 5, replace=False)
plt.figure(figsize=(10,2))
for i, idx in enumerate(idxs):
    img = X_test[idx].reshape(28,28)
    pred = np.argmax(trained_model.predict(img.reshape(1,784)), axis=1)[0]
    pred_name = LABEL_NAMES[pred]
    true_name = LABEL_NAMES[y_test[idx]]
    plt.subplot(1,5,i+1)
    plt.imshow(img, cmap='gray')
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}")
    plt.axis('off')
plt.show()